# Health Insurance: Charge Classification

Educational analysis of high/low charges, not a recommendation of actual insurance plans. Run the setup in the repository README first, then run these cells in order. The original experiments remain in `archive/`.

In Colab: run the setup cell, then **Runtime → Run all**. You can save a personal copy to Drive. The setup reuses an existing checkout without overwriting edits.


In [ ]:
# In Colab, fetch the project files and install dependencies in this runtime.
# Locally, this cell does nothing; follow README setup first.
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    checkout = Path('/content/Health-Insurance-Recommendation-System')
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/hrnareshabd/Health-Insurance-Recommendation-System.git',
                        str(checkout)], check=True)
    os.chdir(checkout)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '-r', 'requirements-notebook.txt'], check=True)
    print('Project ready. If Colab asks for a session restart, restart and run all cells again.')


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'insurance_model').exists():
    ROOT = ROOT.parent
if not (ROOT / 'insurance_model').exists():
    raise FileNotFoundError('Start Jupyter from the repository root or notebooks folder.')
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from insurance_model.pipeline import load_data, split_data, train, predict_profile


## Explore the training partition

Duplicates are removed before splitting. Exploratory plots below use training rows to keep test data out of model development. The high-charge threshold comes only from training charges.


In [ ]:
data = load_data(ROOT / 'Insurance.csv')
x_train, x_test, y_train, y_test, threshold = split_data(data)
print(f'Unique rows: {len(data)}; train: {len(x_train)}; test: {len(x_test)}')
print(f'Training median charge threshold: {threshold:.2f}')
display(x_train.describe(include='all'))
data.loc[x_train.index, 'charges'].plot.hist(bins=30, title='Training charges')
plt.axvline(threshold, color='red', linestyle='--', label='Training median')
plt.legend()
plt.show()


## Select using cross-validation, then evaluate

Default: dummy baseline, logistic regression, random forest. To compare all seven learned classifiers, install `requirements-all-models.txt` and set `all_models=True`. CV values are selection scores. The test is used only after selecting the winner.


In [ ]:
artifact, report = train(ROOT / 'Insurance.csv', all_models=False)
display(pd.DataFrame(report['cv_comparison']))
print('Selected model:', report['selected_model'])
display(pd.Series(report['test_metrics']))


## Explore feature influence

Permutation importance measures how shuffling each input affects F1. Here it uses training data for exploratory interpretation, so it is not a generalization estimate and does not establish causality.


In [ ]:
from sklearn.inspection import permutation_importance
importance = permutation_importance(artifact['pipeline'], x_train, y_train, scoring='f1', n_repeats=5, random_state=42)
summary = pd.DataFrame({'feature': x_train.columns, 'mean_f1_decrease': importance.importances_mean, 'std': importance.importances_std}).sort_values('mean_f1_decrease')
display(summary)
summary.plot.barh(x='feature', y='mean_f1_decrease', legend=False, title='Training permutation importance')
plt.show()


## Predict one example

This invented profile has no observed charge outcome, so it is a demonstration, not an accuracy test. Model probabilities are uncalibrated.


In [ ]:
import json
profile = json.loads((ROOT / 'examples/profile.json').read_text())
predict_profile(artifact, profile)


## Save the complete pipeline

The artifact contains preprocessing, the model, feature names, and the charge threshold. Only load joblib files you created or trust.


In [ ]:
import joblib
output = ROOT / 'artifacts'
output.mkdir(exist_ok=True)
joblib.dump(artifact, output / 'model.joblib')
(output / 'metrics.json').write_text(json.dumps(report, indent=2) + '\n')
print('Saved model and metrics to', output)


## Contribute

Read [CONTRIBUTING](../CONTRIBUTING.md), [methodology and limitations](../docs/METHODOLOGY.md), and the [roadmap](../docs/ROADMAP.md). Dataset provenance, subgroup evaluation, and calibrated explanations are useful next steps.
